### **Model Training**

#### **Importing The Necessary Modules And Packages**

In [ ]:
# Installing All Required Libraries (Consistent)
%pip install -qq peft transformers kagglehub evaluate trackio

In [ ]:
# Importing The Necessary Modules And Packages
from datasets import DatasetDict, Value
import evaluate
from huggingface_hub import login
from kagglehub import KaggleDatasetAdapter, dataset_load
from kaggle_secrets import UserSecretsClient
import numpy as np
import os
from peft import get_peft_model, LoraConfig, TaskType
import random
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, set_seed, Trainer, TrainingArguments

#### **Setting A Random Seed (Ensure That Results Are Always Consistent When Run)**

We will set a fixed random seed (`SEED = 42`) across multiple libraries, including Python’s random, NumPy and PyTorch to ensure reproducibility of results. By doing so, any randomness in data shuffling, weight initialization or sampling will produce the same outcomes every time the code is run. 

We will also configures PyTorch to use deterministic behavior on GPUs by disabling certain optimizations (`cudnn.benchmark = False`) and enforcing consistent computation paths (`cudnn.deterministic = True`). The `set_seed(SEED)` function further ensures that all relevant components follow the same seed.

In [ ]:
# Setting A Random Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)           # CPU
torch.cuda.manual_seed(SEED)      # Current GPU
torch.cuda.manual_seed_all(SEED)  # All GPUs (if multiple)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
set_seed(SEED)

#### **Model Fine-Tuning Configuration (Classification Setup)**

Next, we will define the general configuration settings for a BERT fine-tuning pipeline. It specifies the dataset source (`bingxuanchia/dsa4262-finetuning-exploration`) and the pre-trained model (`google-bert/bert-base-cased`) which will be used. The setup allows for two possible fine-tuning approaches: regression or classification. In this case, classification is selected, which determines several downstream parameters. 

The evaluation metric is set to accuracy (instead of MSE for regression), and the number of output classes is defined as 4 (instead of 1 for regression). It also constructs dynamic names for saving and uploading the fine-tuned model, sets a directory (`weights`) for storing trained model parameters and includes tracking configurations (`TRACKIO_SPACE_ID` and `TRACKIO_PROJECT`) for experiment logging.

In [ ]:
# General configurations

# Dataset source (regression/classification)
DATASET_PATH = "bingxuanchia/dsa4262-finetuning-exploration"
# Pre-trained BERT model
MODEL_NAME = "google-bert/bert-base-cased"  

# Selected method: classification
FINETUNING_METHOD = "classification"
# Name for saved model
FINETUNED_MODEL_NAME = f"v2-heladepdet-bert-finetuned-{FINETUNING_METHOD}"
# Hugging Face repo ID
FINETUNED_MODEL_REPO_ID = f"chiabingxuan/{FINETUNED_MODEL_NAME}"  

# Use MSE for regression
if FINETUNING_METHOD == "regression":
    FINETUNING_METRIC = "mse"  
# Use accuracy for classification
else:
    FINETUNING_METRIC = "accuracy"  

# Directory to store model weights
MODEL_WEIGHTS_PATH = "weights"  

# Single output for regression
if FINETUNING_METHOD == "regression":
    NUM_CLASSES = 1
# Number of classes for classification
else:
    NUM_CLASSES = 4 

# Tracking workspace ID
TRACKIO_SPACE_ID = "chiabingxuan/DSA4262-FineTuning"
# Project name for experiment tracking
TRACKIO_PROJECT = "bert-finetuning"  

#### **Training Configuration and Hyperparameters**

We now defines the training setup for fine-tuning the BERT model using the `TrainingArguments` class from Hugging Face Transformers. It specifies where the trained model outputs will be saved, along with key hyperparameters that control the learning process such as the learning rate (1e-4), number of training epochs (10) and batch sizes for both training and evaluation.

The configuration uses a step-based evaluation and checkpointing strategy, meaning the model is evaluated and saved every 250 steps, allowing for more frequent monitoring of performance during training. It also ensures that the best-performing model (based on the chosen metric) is loaded at the end. Reproducibility is maintained through a fixed random seed. 

Additionally, the setup integrates with the Hugging Face Hub to automatically push the trained model and enables experiment tracking via **Trackio**, logging progress at regular intervals and associating the run with a specific project and workspace.

In [ ]:
# Training configurations
TRAIN_CONFIG = TrainingArguments(
    # Directory to save model outputs
    output_dir=os.path.join(MODEL_WEIGHTS_PATH, FINETUNED_MODEL_NAME), 

    learning_rate=1e-4,                   # Learning rate for optimizer
    num_train_epochs=10,                  # Number of training epochs
    per_device_train_batch_size=16,       # Training batch size per device
    per_device_eval_batch_size=16,        # Evaluation batch size per device
    eval_strategy="steps",                # Evaluate at fixed step intervals
    eval_steps=250,                       # Evaluation frequency (steps)
    save_strategy="steps",                # Save checkpoints at fixed step intervals
    save_steps=250,                       # Checkpoint saving frequency (steps)
    load_best_model_at_end=True,          # Load best model after training
    seed=SEED,                            # Set random seed for reproducibility

    push_to_hub=True,                     # Upload model to Hugging Face Hub

    logging_steps=250,                    # Logging frequency (steps)
    report_to=["trackio"],                # Enable Trackio experiment tracking
    trackio_space_id=TRACKIO_SPACE_ID,    # Trackio workspace ID
    project=TRACKIO_PROJECT,              # Trackio project name
    run_name=FINETUNED_MODEL_NAME         # Name of the training run
)

#### **LoRA (Low-Rank Adaptation) Configuration**

Additionally, we will define the configuration for applying **LoRA (Low-Rank Adaptation)** during the fine-tuning process, which is an efficient parameter-efficient fine-tuning (PEFT) technique. Instead of updating all the weights in the large BERT model, LoRA introduces small trainable low-rank matrices into selected layers, significantly reducing the number of parameters that need to be updated. 

Here, LoRA is enabled (`LORA_ENABLED = True`), meaning the model will use this lightweight adaptation approach. The rank `r=16` controls the size of the low-rank decomposition while `lora_alpha=16` scales the updates and `lora_dropout=0.1` helps prevent overfitting during training. The `bias="none"` setting indicates that bias terms are not adapted and `task_type=TaskType.SEQ_CLS` specifies that the model is being fine-tuned for a sequence classification task. 

In [ ]:
# LoRA configurations - Enable LoRA for parameter-efficient fine-tuning
LORA_ENABLED = True 

LORA_CONFIG = LoraConfig(
    r=16,                         # Rank of low-rank adaptation matrices
    lora_alpha=16,                # Scaling factor for LoRA updates
    lora_dropout=0.1,             # Dropout rate for regularization
    bias="none",                  # Do not adapt bias parameters
    task_type=TaskType.SEQ_CLS    # Task type: sequence classification
)

#### **Dataset Loading from Kaggle**

The function `load_kaggle_dataset` is responsible for loading the dataset from Kaggle and organizing it into a structured format suitable for model training and evaluation. It iterates through three predefined data splits: train, validation (val) and test. It then loads each corresponding CSV file using the `dataset_load` function with the `KaggleDatasetAdapter` configured for Hugging Face compatibility. 

Each split is stored in a dictionary, which is then converted into a `DatasetDict`, a standardized format used in the Hugging Face ecosystem for handling multiple dataset splits efficiently. A confirmation message is printed to indicate successful loading and the complete dataset object is returned. This modular design ensures that data loading is clean, reusable and consistent across experiments.

In [ ]:
# Dataset Loading from Kaggle
def load_kaggle_dataset(dataset_path):
    # Initialize dictionary to store dataset splits
    dataset = dict() 

    # Iterate over dataset splits
    for phase in ["train", "val", "test"]:  
        phase_data = dataset_load(
            KaggleDatasetAdapter.HUGGING_FACE,     # Use Hugging Face adapter
            dataset_path,                          # Kaggle dataset path
            f"{phase}.csv")                        # Load corresponding CSV file
        dataset[phase] = phase_data                # Store split data in dictionary

    # Convert to Hugging Face DatasetDict format
    dataset = DatasetDict(dataset)
    # Confirmation message
    print(f"Dataset loaded from {dataset_path} on Kaggle!\n") 
    return dataset 

#### **Dataset Preprocessing and Tokenization**

Next, we will handle the preprocessing of the dataset by converting raw text into a format suitable for input into a BERT model. 

1. The `preprocess_function` applies a tokenizer to the text data, transforming each input sentence into token IDs while ensuring consistent input length through padding (`padding="max_length"`) and truncation of longer sequences.

2. The `preprocess_dataset` function initializes the tokenizer using a specified pre-trained model and applies the preprocessing function across the entire dataset using the Hugging Face map method, which efficiently processes data in batches. It also removes the original text column after tokenization to keep only model-ready features.

Additionally, if the task is regression, the label column is explicitly cast to a float type to ensure compatibility with regression objectives. The function prints a confirmation message along with the processed dataset structure and returns both the processed dataset and tokenizer.

In [ ]:
# Defining The Preprocess Function
def preprocess_function(tokeniser, example, text_column):
    # Tokenize text with padding/truncation
    examples = tokeniser(example[text_column], padding="max_length", truncation=True)
    # Return tokenized output
    return examples
    
# Defining The Preprocess Dataset
def preprocess_dataset(dataset, tokeniser_model_name, text_column):
    # Load pre-trained tokenizer
    tokeniser = AutoTokenizer.from_pretrained(tokeniser_model_name)

    dataset = dataset.map(
        # Apply tokenization
        lambda example: preprocess_function(tokeniser, example, text_column),
        # Remove raw text column after processing
        remove_columns=[text_column], 
        # Process data in batches for efficiency
        batched=True)
    
    if FINETUNING_METHOD == "regression":
        # Cast labels to float for regression
        dataset = dataset.cast_column("labels", Value("float32"))

    # Confirmation message
    print(f"Pre-processing completed with {tokeniser_model_name}!\n") 
    # Display dataset structure
    print(dataset) 
    # Return processed dataset and tokenizer
    return dataset, tokeniser 

#### **Evaluation Metrics and Performance Computation**

Afterwards, we define how model performance is evaluated during fine-tuning for both regression and classification tasks. Two separate metric computation functions (`compute_metrics_regression` and `compute_metrics_classification`) are implemented to handle the differences in output formats between these tasks. 

* For **regression**, the model outputs continuous values so both predictions and labels are reshaped into one-dimensional arrays before computing evaluation metrics such as Mean Squared Error (MSE).

* For **classification**, the model produces logits (raw scores for each class), which are converted into predicted class labels using argmax, selecting the class with the highest score.

The appropriate function is then dynamically selected based on the chosen fine-tuning method (classification in this case), ensuring that evaluation is correctly aligned with the task type. This modular approach allows seamless switching between regression and classification while maintaining proper metric computation.

In [ ]:
# Metrics to track performance during fine-tunings

# Defining Compute Metrics Regression Function
def compute_metrics_regression(eval_metric, eval_pred):
    predictions, labels = eval_pred           # Unpack predictions and true labels
    predictions = predictions.reshape(-1)     # Flatten predictions to 1D
    labels = labels.reshape(-1)               # Flatten labels to 1D
    return eval_metric.compute(predictions=predictions, references=labels)

# Defining Compute Metrics Classification Function
def compute_metrics_classification(eval_metric, eval_pred):
    logits, labels = eval_pred                # Unpack logits and true labels
    predictions = np.argmax(logits, axis=-1)  # Convert logits to class predictions
    return eval_metric.compute(predictions=predictions, references=labels)

# Select appropriate metric function based on task

# Use regression metrics
if FINETUNING_METHOD == "regression":
    chosen_compute_metrics = compute_metrics_regression
# Use classification metrics
else:
    chosen_compute_metrics = compute_metrics_classification

#### **Reproducibility and Device Setup**

We will define two utility functions to ensure consistent experiment results and to configure the computation device for training. 

1. The `set_seeds` function sets a fixed random seed across multiple libraries, Python’s random, NumPy and PyTorch (both CPU and GPU) to guarantee reproducibility, meaning the same inputs will produce identical results across runs. It also enforces deterministic behavior in CUDA operations by disabling certain performance optimizations that introduce randomness.

2. The `check_device` function detects whether a CUDA-enabled GPU is available and if so, selects it as the computation device while printing useful information such as the GPU name and memory capacity. If no GPU is available, it defaults to using the CPU.

In [ ]:
def set_seeds(seed):
    random.seed(seed)                          # Set Python random seed
    np.random.seed(seed)                       # Set NumPy random seed
    torch.manual_seed(seed)                    # Set PyTorch CPU seed
    torch.cuda.manual_seed(seed)               # Set current GPU seed
    torch.cuda.manual_seed_all(seed)           # Set all GPU seeds
    torch.backends.cudnn.deterministic = True  # Ensure deterministic GPU operations
    torch.backends.cudnn.benchmark = False     # Disable performance optimizations for reproducibility
    set_seed(seed)                             # Set seed for Hugging Face

def check_device():
    # Check if GPU is available
    if torch.cuda.is_available(): 
        # Use GPU
        device = "cuda"
        # Print GPU name
        print(f"Using CUDA GPU: {torch.cuda.get_device_name()}") 
        # Print GPU memory
        print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB\n")
        
    # Fallback to CPU
    else:
        device = "cpu"
        # Print CPU usage
        print("Using CPU\n") 
    return device

#### **Model Loading and LoRA Integration**

Next, we load the pre-trained BERT model and optionally integrate LoRA (Low-Rank Adaptation) for parameter-efficient fine-tuning. The `load_model` function first loads a pre-trained **AutoModelForSequenceClassification** model with the specified number of output classes, automatically mapping it to the best available device (CPU or GPU). 

If LoRA is enabled (`lora_enabled=True`), the model is wrapped with the LoRA configuration, allowing only a small subset of trainable parameters to be updated during fine-tuning which reduces computational cost while maintaining performance. The function prints the LoRA trainable parameters and the full model architecture.

In [ ]:
# Defining The Load Model Function
def load_model(model_name, num_classes, lora_enabled):
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_classes)     # Set output layer size

    # Apply LoRA if enabled
    if lora_enabled:  
        # Wrap model with LoRA
        model = get_peft_model(model, LORA_CONFIG)
        # Indicate LoRA activation
        print(f"LoRA enabled:") 
        # Display trainable parameters
        model.print_trainable_parameters() 
        print()

    # Print model structure
    print(f"Model architecture:")  
    # Display full model
    print(model) 
    print()
    # Return the configured model
    return model 

#### **Model Fine-Tuning and Hub Upload**

After defining the `load_model` function, we will orchestrate the fine-tuning of the pre-trained BERT model on the provided dataset and handle uploading the resulting model to the Hugging Face Hub. 

The `finetune_model` function ensures that the directory for saving model weights exists, then loads the evaluation metric (accuracy for classification or MSE for regression) using the evaluate library. A `Trainer` object is created with the model, training configurations, training and validation datasets and the appropriate metric computation function. The `train()` method performs the fine-tuning process according to the hyperparameters defined earlier. 

After training, both the fine-tuned model and tokenizer are pushed to the Hugging Face Hub under the specified repository ID, making them publicly accessible for reuse. The function finally returns the trained model, ready for inference and further evaluation. This setup centralizes the fine-tuning workflow while ensuring reproducibility, monitoring and easy sharing.

In [ ]:
def finetune_model(model, model_name, tokeniser, finetuned_model_name, repo_id, dataset, model_weights_path):
    # Ensure weight-saving directory exists
    os.makedirs(model_weights_path, exist_ok=True)
    # Load evaluation metric
    metric = evaluate.load(FINETUNING_METRIC)

    # Print Status message
    print(f"Fine-tuning {model_name}...") 
    trainer = Trainer(
        model=model,                        # Model to fine-tune
        args=TRAIN_CONFIG,                  # Training hyperparameters
        train_dataset=dataset["train"],     # Training dataset
        eval_dataset=dataset["val"],        # Validation dataset
        # Compute task-specific metrics
        compute_metrics=lambda eval_pred: chosen_compute_metrics(metric, eval_pred))
    
    # Perform fine-tuning
    trainer.train()
    print("Fine-tuning completed!")

    # Upload fine-tuned model and tokenizer to Hugging Face Hub
    trainer.push_to_hub()                  # Push model to Hub
    tokeniser.push_to_hub(repo_id)         # Push tokenizer to Hub
    print(f"Fine-tuned model {finetuned_model_name} and tokeniser uploaded to Hugging Face Hub!")
    return model

#### **Creating End-to-End Fine-Tuning Pipeline**

We can now define and implement the complete workflow for fine-tuning the BERT model on the Kaggle dataset and pushing the results to Hugging Face Hub. 

The `run_finetuning_pipeline` function starts by authenticating with Hugging Face using a provided token and then sets a fixed random seed to ensure reproducibility. The available computation device (GPU or CPU) is detected and displayed. The dataset is then loaded from Kaggle, followed by tokenization and preprocessing to prepare it for the model. 

Next, the pre-trained BERT model is loaded, optionally with LoRA integration for parameter-efficient fine-tuning. The model is then fine-tuned on the training set with evaluation on the validation set, using the appropriate metrics for classification. Finally, the fine-tuned model and tokenizer are uploaded to Hugging Face Hub. 

This modular pipeline ensures a structured, reproducible, and end-to-end process for model training and sharing.

In [ ]:
def run_finetuning_pipeline(hf_token):
    login(hf_token)                       # Authenticate with Hugging Face
    set_seeds(SEED)                       # Set random seed for reproducibility

    device = check_device()               # Detect and print computation device

    dataset = load_kaggle_dataset(dataset_path=DATASET_PATH)    # Load dataset from Kaggle
    
    dataset, tokeniser = preprocess_dataset(
        dataset=dataset,                  # Dataset to preprocess
        tokeniser_model_name=MODEL_NAME,  # Pre-trained tokenizer
        text_column="text")               # Column containing text data

    model = load_model(
        model_name=MODEL_NAME,            # Pre-trained model name
        num_classes=NUM_CLASSES,          # Number of output classes
        lora_enabled=LORA_ENABLED)        # Apply LoRA if enabled
    
    model_ft = finetune_model(
        model=model,                                     # Model to fine-tune
        model_name=MODEL_NAME,                           # Base model name
        tokeniser=tokeniser,                             # Tokenizer for preprocessing
        finetuned_model_name=FINETUNED_MODEL_NAME,       # Name for fine-tuned model
        repo_id=FINETUNED_MODEL_REPO_ID,                 # Hugging Face repo ID
        dataset=dataset,                                 # Preprocessed dataset
        model_weights_path=MODEL_WEIGHTS_PATH)           # Directory to save model weights

# Retrieve Hugging Face token from secrets
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
# Execute end-to-end fine-tuning
run_finetuning_pipeline(hf_token=hf_token)